# NFL Next Gen Stats Receiving Data Acquisition

## Purpose

This notebook acquires a secondary dataset for our CS 418 football project.

### Research Question

How much does average WR separation correlate with receiving yardage?

### Dataset

NFL Next Gen Stats receiving data accessed through `nflreadpy`.

This dataset contains advanced receiving metrics such as average separation, average cushion, targets, receptions, catch percentage, receiving yards, and yards after catch.

For this project, we will primarily use 2025 regular-season data and focus on wide receivers.

In [1]:
%pip install nflreadpy polars

  Using cached nflreadpy-0.1.5-py3-none-any.whl.metadata (7.5 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached nflreadpy-0.1.5-py3-none-any.whl (30 kB)
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ---------------------------------------- 865.8/865.8 kB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/51.3 MB ? eta -:--:--
   - -------------------------------------- 2.4/51.3 MB 11.2 MB/s eta 0:00:05
   --- ------------------------------------ 4.7/51.3 MB 11.0 MB/s eta 0:00:05
   ----- ---------------------------------- 6.6/51.3 MB 10.6 MB/s eta 0:00:05
   ------ --------------------------------- 8.4/51.3 MB 10.0 MB/s eta 0:00:05
   -------- ------------------------------- 10.7/51.3 MB 10.0 MB/s eta 0:00:05
   ---------- ----------------------------- 12.8/51.3 MB 10.2 MB/s eta 0:00:04
   ----------- ----------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import nflreadpy as nfl
import polars as pl

print("nflreadpy loaded successfully")
print("Polars version:", pl.__version__)

nflreadpy loaded successfully
Polars version: 1.44.2


In [2]:
ngs_receiving = nfl.load_nextgen_stats(
    seasons=2025,
    stat_type="receiving"
)

print("Shape:", ngs_receiving.shape)

Shape: (1402, 23)


In [3]:
print(ngs_receiving.columns)

['season', 'season_type', 'week', 'player_display_name', 'player_position', 'team_abbr', 'avg_cushion', 'avg_separation', 'avg_intended_air_yards', 'percent_share_of_intended_air_yards', 'receptions', 'targets', 'catch_percentage', 'yards', 'rec_touchdowns', 'avg_yac', 'avg_expected_yac', 'avg_yac_above_expectation', 'player_gsis_id', 'player_first_name', 'player_last_name', 'player_jersey_number', 'player_short_name']


In [4]:
ngs_receiving.head()

season,season_type,week,player_display_name,player_position,team_abbr,avg_cushion,avg_separation,avg_intended_air_yards,percent_share_of_intended_air_yards,receptions,targets,catch_percentage,yards,rec_touchdowns,avg_yac,avg_expected_yac,avg_yac_above_expectation,player_gsis_id,player_first_name,player_last_name,player_jersey_number,player_short_name
i32,str,i32,str,str,str,f64,f64,f64,f64,i32,i32,f64,i32,i32,f64,f64,f64,str,str,str,i32,str
2025,"""REG""",0,"""Marvin Mims""","""WR""","""DEN""",7.845641,4.516368,9.053333,10.535467,37,51,72.54902,322,1,6.492162,6.026702,0.46546,"""00-0038976""","""Marvin""","""Mims""",19,"""M.Mims"""
2025,"""REG""",0,"""Tre Tucker""","""WR""","""LV""",7.652759,3.560551,10.3425,29.114191,57,92,61.956522,696,5,5.240175,4.870006,0.370169,"""00-0038563""","""Tre""","""Tucker""",1,"""T.Tucker"""
2025,"""REG""",0,"""Luther Burden""","""WR""","""CHI""",7.328824,4.624625,7.280667,9.73822,47,60,78.333333,652,2,7.39766,6.465753,0.931907,"""00-0040735""","""Luther""","""Burden""",10,"""L.Burden"""
2025,"""REG""",0,"""Jordan Addison""","""WR""","""MIN""",7.191711,2.872843,14.387848,29.871724,42,79,53.164557,610,3,3.604048,3.000354,0.603694,"""00-0038994""","""Jordan""","""Addison""",3,"""J.Addison"""
2025,"""REG""",0,"""Josh Downs""","""WR""","""IND""",7.172917,2.631882,7.794659,15.970282,58,88,65.909091,559,4,2.637241,2.512249,0.124993,"""00-0038997""","""Joshua""","""Downs""",2,"""J.Downs"""


## Filter to Regular-Season Wide Receivers

The Next Gen Stats dataset contains both summary rows and weekly observations.

For our research question, we keep:
- 2025 season
- regular season games
- wide receivers
- weekly observations only (`week > 0`)

In [5]:
wr_receiving = (
    ngs_receiving
    .filter(
        (pl.col("season") == 2025) &
        (pl.col("season_type") == "REG") &
        (pl.col("player_position") == "WR") &
        (pl.col("week") > 0)
    )
)

print("Filtered shape:", wr_receiving.shape)

wr_receiving.head()

Filtered shape: (888, 23)


season,season_type,week,player_display_name,player_position,team_abbr,avg_cushion,avg_separation,avg_intended_air_yards,percent_share_of_intended_air_yards,receptions,targets,catch_percentage,yards,rec_touchdowns,avg_yac,avg_expected_yac,avg_yac_above_expectation,player_gsis_id,player_first_name,player_last_name,player_jersey_number,player_short_name
i32,str,i32,str,str,str,f64,f64,f64,f64,i32,i32,f64,i32,i32,f64,f64,f64,str,str,str,i32,str
2025,"""REG""",1,"""Emeka Egbuka""","""WR""","""TB""",9.023333,2.586366,12.431667,24.3591,4,6,66.666667,67,2,4.3025,6.822941,-2.520441,"""00-0040129""","""Emeka""","""Egbuka""",2,"""E.Egbuka"""
2025,"""REG""",1,"""Calvin Ridley""","""WR""","""TEN""",8.785714,4.541601,8.30625,24.382637,4,8,50.0,27,0,3.25,4.041631,-0.791631,"""00-0034837""","""Calvin""","""Ridley""",0,"""C.Ridley"""
2025,"""REG""",1,"""Josh Palmer""","""WR""","""BUF""",8.015556,2.449627,13.424444,28.570077,5,9,55.555556,61,0,2.416,2.14643,0.26957,"""00-0036988""","""Josh""","""Palmer""",5,"""J.Palmer"""
2025,"""REG""",1,"""Jameson Williams""","""WR""","""DET""",7.95,3.731178,5.56,16.418616,4,5,80.0,23,0,2.0675,1.881923,0.185577,"""00-0037240""","""Jameson""","""Williams""",1,"""J.Williams"""
2025,"""REG""",1,"""Travis Hunter""","""WR""","""JAX""",7.875,3.132607,6.6375,25.485961,6,8,75.0,33,0,3.951667,4.426598,-0.474931,"""00-0040718""","""Travis""","""Hunter""",12,"""T.Hunter"""


## Select Relevant Columns

For the analysis, we keep the player identifiers, team/week information,
average separation metrics, receiving volume, receiving yards, and YAC metrics.

These variables will allow us to study the relationship between WR separation
and receiving production.

In [6]:
wr_receiving = wr_receiving.select([
    "player_gsis_id",
    "player_display_name",
    "player_position",
    "team_abbr",
    "season",
    "week",
    "avg_cushion",
    "avg_separation",
    "avg_intended_air_yards",
    "targets",
    "receptions",
    "catch_percentage",
    "yards",
    "rec_touchdowns",
    "avg_yac",
    "avg_expected_yac",
    "avg_yac_above_expectation"
])

print("Shape after selecting columns:", wr_receiving.shape)

wr_receiving.head()

Shape after selecting columns: (888, 17)


player_gsis_id,player_display_name,player_position,team_abbr,season,week,avg_cushion,avg_separation,avg_intended_air_yards,targets,receptions,catch_percentage,yards,rec_touchdowns,avg_yac,avg_expected_yac,avg_yac_above_expectation
str,str,str,str,i32,i32,f64,f64,f64,i32,i32,f64,i32,i32,f64,f64,f64
"""00-0040129""","""Emeka Egbuka""","""WR""","""TB""",2025,1,9.023333,2.586366,12.431667,6,4,66.666667,67,2,4.3025,6.822941,-2.520441
"""00-0034837""","""Calvin Ridley""","""WR""","""TEN""",2025,1,8.785714,4.541601,8.30625,8,4,50.0,27,0,3.25,4.041631,-0.791631
"""00-0036988""","""Josh Palmer""","""WR""","""BUF""",2025,1,8.015556,2.449627,13.424444,9,5,55.555556,61,0,2.416,2.14643,0.26957
"""00-0037240""","""Jameson Williams""","""WR""","""DET""",2025,1,7.95,3.731178,5.56,5,4,80.0,23,0,2.0675,1.881923,0.185577
"""00-0040718""","""Travis Hunter""","""WR""","""JAX""",2025,1,7.875,3.132607,6.6375,8,6,75.0,33,0,3.951667,4.426598,-0.474931


## Check for Missing Values

Before analyzing the dataset, we check whether important variables contain
missing values.

In [7]:
wr_receiving.null_count()

player_gsis_id,player_display_name,player_position,team_abbr,season,week,avg_cushion,avg_separation,avg_intended_air_yards,targets,receptions,catch_percentage,yards,rec_touchdowns,avg_yac,avg_expected_yac,avg_yac_above_expectation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,2,2


In [8]:
important_nulls = wr_receiving.select([
    "avg_separation",
    "targets",
    "receptions",
    "yards",
    "avg_yac",
    "avg_expected_yac",
    "avg_yac_above_expectation"
]).null_count()

important_nulls

avg_separation,targets,receptions,yards,avg_yac,avg_expected_yac,avg_yac_above_expectation
u32,u32,u32,u32,u32,u32,u32
0,0,0,1,1,2,2


## Prepare Data for Analysis

The main research question requires average WR separation and receiving yardage.

There are no missing values for `avg_separation` and only one missing value
for `yards`. We remove rows missing either of these variables for the main
analysis rather than dropping observations because of missing YAC metrics.

In [10]:
wr_analysis = wr_receiving.filter(
    pl.col("avg_separation").is_not_null() &
    pl.col("yards").is_not_null()
)

print("Original shape:", wr_receiving.shape)
print("Analysis shape:", wr_analysis.shape)

Original shape: (888, 17)
Analysis shape: (887, 17)


## Create Receiving Efficiency Variables

Total receiving yards are influenced by how many opportunities a receiver gets.

To compare receiving efficiency as well as total production, we calculate:

- yards per target
- yards per reception

In [11]:
wr_analysis = wr_analysis.with_columns([
    pl.when(pl.col("targets") > 0)
      .then(pl.col("yards") / pl.col("targets"))
      .otherwise(None)
      .alias("yards_per_target"),

    pl.when(pl.col("receptions") > 0)
      .then(pl.col("yards") / pl.col("receptions"))
      .otherwise(None)
      .alias("yards_per_reception")
])

wr_analysis.select([
    "player_display_name",
    "week",
    "avg_separation",
    "targets",
    "receptions",
    "yards",
    "yards_per_target",
    "yards_per_reception"
]).head(10)

player_display_name,week,avg_separation,targets,receptions,yards,yards_per_target,yards_per_reception
str,i32,f64,i32,i32,i32,f64,f64
"""Emeka Egbuka""",1,2.586366,6,4,67,11.166667,16.75
"""Calvin Ridley""",1,4.541601,8,4,27,3.375,6.75
"""Josh Palmer""",1,2.449627,9,5,61,6.777778,12.2
"""Jameson Williams""",1,3.731178,5,4,23,4.6,5.75
"""Travis Hunter""",1,3.132607,8,6,33,4.125,5.5
"""Garrett Wilson""",1,3.78192,9,7,95,10.555556,13.571429
"""Keon Coleman""",1,3.663162,11,8,112,10.181818,14.0
"""Khalil Shakir""",1,3.543695,9,6,64,7.111111,10.666667
"""Jakobi Meyers""",1,2.951062,10,8,97,9.7,12.125


## Summary Statistics

Before calculating correlations, we examine the distributions of the main
variables used in the analysis.

The key variables are:

- average separation
- receiving yards
- yards per target
- yards per reception
- targets
- receptions

In [12]:
wr_analysis.select([
    "avg_separation",
    "yards",
    "yards_per_target",
    "yards_per_reception",
    "targets",
    "receptions"
]).describe()

statistic,avg_separation,yards,yards_per_target,yards_per_reception,targets,receptions
str,f64,f64,f64,f64,f64,f64
"""count""",887.0,887.0,887.0,887.0,887.0,887.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",2.848074,61.193912,7.910985,12.678648,7.72717,4.827508
"""std""",0.969516,35.607952,3.904626,5.446103,2.578338,2.161276
"""min""",0.743045,-2.0,-0.4,-2.0,5.0,1.0
"""25%""",2.168218,34.0,5.0,9.0,6.0,3.0
"""50%""",2.684806,55.0,7.4,12.0,7.0,4.0
"""75%""",3.406147,83.0,10.277778,15.5,9.0,6.0
"""max""",6.945923,225.0,25.2,43.0,23.0,16.0


## Correlation Between WR Separation and Receiving Production

We calculate Pearson correlation coefficients to examine the linear
relationship between average WR separation and different measures of
receiving production.

We compare average separation with:

- total receiving yards
- yards per target
- yards per reception

Correlation measures association and does not imply causation.

In [13]:
correlations = wr_analysis.select([
    pl.corr("avg_separation", "yards")
      .alias("separation_vs_yards"),

    pl.corr("avg_separation", "yards_per_target")
      .alias("separation_vs_yards_per_target"),

    pl.corr("avg_separation", "yards_per_reception")
      .alias("separation_vs_yards_per_reception")
])

correlations

separation_vs_yards,separation_vs_yards_per_target,separation_vs_yards_per_reception
f64,f64,f64
0.017027,0.083483,-0.129256


## Initial Findings

Using 887 valid 2025 regular-season WR-week observations, the initial Pearson
correlations between average separation and receiving production are weak.

- Average separation vs. receiving yards: approximately 0.017
- Average separation vs. yards per target: approximately 0.083
- Average separation vs. yards per reception: approximately -0.129

These results suggest that average WR separation alone does not have a strong
linear relationship with receiving production at the weekly level.

This is an exploratory result and does not imply that separation has no effect
on receiving performance. Other factors such as target volume, quarterback
play, route type, defensive coverage, and air yards may also affect receiving
production.

## Save Cleaned Dataset

The cleaned WR dataset is saved as a CSV so it can be reused in later stages
of the project without repeating the acquisition and cleaning steps.

In [14]:
wr_analysis.write_csv("nextgen_wr_receiving_2025.csv")

print("Saved nextgen_wr_receiving_2025.csv")
print("Final dataset shape:", wr_analysis.shape)

Saved nextgen_wr_receiving_2025.csv
Final dataset shape: (887, 19)
